In [0]:
from pyspark.sql.functions import col, trim, to_timestamp, to_date, when, lit, lower
from pyspark.sql import functions as F
import builtins

spark.sql("DROP TABLE IF EXISTS social_catalog.silver.tweets_table")
spark.sql("DROP TABLE IF EXISTS social_catalog.silver.sentiment_table")
spark.sql("DROP TABLE IF EXISTS social_catalog.silver.trends_table")
spark.sql("DROP TABLE IF EXISTS social_catalog.silver.user_metadata_table")
spark.sql("DROP TABLE IF EXISTS social_catalog.silver.valid_table")
spark.sql("CREATE SCHEMA IF NOT EXISTS social_catalog.silver")

print("Silver schema ready")

tweets_bronze    = spark.table("social_catalog.bronze.tweets_table")
sentiment_bronze = spark.table("social_catalog.bronze.sentiment_table")
trends_bronze    = spark.table("social_catalog.bronze.trends_table")
users_bronze     = spark.table("social_catalog.bronze.user_metadata_table")
valid_bronze     = spark.table("social_catalog.bronze.valid_table")

print("BRONZE ROW COUNTS")
print("tweets_table        : " + str(tweets_bronze.count()))
print("sentiment_table     : " + str(sentiment_bronze.count()))
print("trends_table        : " + str(trends_bronze.count()))
print("user_metadata_table : " + str(users_bronze.count()))
print("valid_table         : " + str(valid_bronze.count()))

r = builtins.round

avg_likes_tweets       = r(tweets_bronze.agg(F.avg("likes")).collect()[0][0] or 0)
avg_retweets_tweets    = r(tweets_bronze.agg(F.avg("retweets")).collect()[0][0] or 0)
avg_replies_tweets     = r(tweets_bronze.agg(F.avg("replies")).collect()[0][0] or 0)
avg_impressions_tweets = r(tweets_bronze.agg(F.avg("impressions")).collect()[0][0] or 0)
avg_engagement_tweets  = r(tweets_bronze.agg(F.avg("engagement")).collect()[0][0] or 0)

avg_likes_sent         = r(sentiment_bronze.agg(F.avg("likes")).collect()[0][0] or 0)
avg_impressions_sent   = r(sentiment_bronze.agg(F.avg("impressions")).collect()[0][0] or 0)
avg_engagement_sent    = r(sentiment_bronze.agg(F.avg("engagement_count")).collect()[0][0] or 0)
avg_sentiment          = r(sentiment_bronze.agg(F.avg("sentiment_score")).collect()[0][0] or 0, 4)
avg_positive           = r(sentiment_bronze.agg(F.avg("positive_score")).collect()[0][0] or 0, 4)
avg_negative           = r(sentiment_bronze.agg(F.avg("negative_score")).collect()[0][0] or 0, 4)
avg_neutral            = r(sentiment_bronze.agg(F.avg("neutral_score")).collect()[0][0] or 0, 4)

avg_tweet_volume       = r(trends_bronze.agg(F.avg("tweet_volume")).collect()[0][0] or 0)
avg_mention_count      = r(trends_bronze.agg(F.avg("mention_count")).collect()[0][0] or 0)
avg_retweet_count      = r(trends_bronze.agg(F.avg("retweet_count")).collect()[0][0] or 0)
avg_trend_score        = r(trends_bronze.agg(F.avg("trend_score")).collect()[0][0] or 0, 4)
avg_sentiment_index    = r(trends_bronze.agg(F.avg("sentiment_index")).collect()[0][0] or 0, 4)
avg_impressions_trends = r(trends_bronze.agg(F.avg("impressions")).collect()[0][0] or 0)
avg_engagement_trends  = r(trends_bronze.agg(F.avg("engagement_count")).collect()[0][0] or 0)

avg_followers          = r(users_bronze.agg(F.avg("followers_count")).collect()[0][0] or 0)
avg_following          = r(users_bronze.agg(F.avg("following_count")).collect()[0][0] or 0)
avg_likes_users        = r(users_bronze.agg(F.avg("likes_count")).collect()[0][0] or 0)
avg_shares             = r(users_bronze.agg(F.avg("shares_count")).collect()[0][0] or 0)
avg_posts              = r(users_bronze.agg(F.avg("posts_count")).collect()[0][0] or 0)

avg_impressions_valid  = r(valid_bronze.agg(F.avg("impressions")).collect()[0][0] or 0)
avg_likes_valid        = r(valid_bronze.agg(F.avg("likes")).collect()[0][0] or 0)
avg_retweets_valid     = r(valid_bronze.agg(F.avg("retweets")).collect()[0][0] or 0)
avg_replies_valid      = r(valid_bronze.agg(F.avg("replies")).collect()[0][0] or 0)
avg_engagement_valid   = r(valid_bronze.agg(F.avg("engagement_count")).collect()[0][0] or 0)
avg_sentiment_valid    = r(valid_bronze.agg(F.avg("sentiment_score")).collect()[0][0] or 0, 4)

topic_list   = ["AI", "Sports", "Finance", "Cloud"]
country_list = ["USA", "UK", "India", "Germany", "Canada"]

print("Averages computed")


before_tweets = tweets_bronze.count()
silver_tweets = (
    tweets_bronze
    .withColumn("tweet_text",
        when(col("tweet_text").isNull(), "No Tweet Content")
        .when(lower(col("tweet_text")) == "null", "No Tweet Content")
        .when(trim(col("tweet_text")) == "", "No Tweet Content")
        .otherwise(trim(col("tweet_text")))
    )
    .withColumn("timestamp",
        to_timestamp(trim(col("timestamp")), "dd-MM-yyyy HH:mm")
    )
    .withColumn("timestamp",
        when(col("timestamp").isNull(),
            F.to_timestamp(F.lit("2025-01-15 12:00"), "yyyy-MM-dd HH:mm")
        ).otherwise(col("timestamp"))
    )
    .withColumn("likes",       when(col("likes").isNull(),       lit(avg_likes_tweets)).otherwise(col("likes")))
    .withColumn("retweets",    when(col("retweets").isNull(),    lit(avg_retweets_tweets)).otherwise(col("retweets")))
    .withColumn("replies",     when(col("replies").isNull(),     lit(avg_replies_tweets)).otherwise(col("replies")))
    .withColumn("impressions", when(col("impressions").isNull(), lit(avg_impressions_tweets)).otherwise(col("impressions")))
    .withColumn("engagement",  when(col("engagement").isNull(),  lit(avg_engagement_tweets)).otherwise(col("engagement")))
    .filter(col("tweet_id").isNotNull())
    .dropDuplicates()
)
after_tweets = silver_tweets.count()
silver_tweets.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.silver.tweets_table")
print("tweets_table before  : " + str(before_tweets))
print("tweets_table after   : " + str(after_tweets))
print("tweets_table removed : " + str(before_tweets - after_tweets))
print()


before_sentiment = sentiment_bronze.count()
silver_sentiment = (
    sentiment_bronze
    .withColumn("topic_category",
        when(col("topic_category").isNull(),
            F.element_at(F.array([lit(t) for t in topic_list]),
                (F.abs(F.hash(col("tweet_id"))) % 4 + 1).cast("int"))
        ).otherwise(trim(col("topic_category")))
    )
    .withColumn("tweet_timestamp",
        to_timestamp(trim(col("tweet_timestamp")), "dd-MM-yyyy HH:mm")
    )
    .withColumn("tweet_timestamp",
        when(col("tweet_timestamp").isNull(),
            F.to_timestamp(F.lit("2025-01-15 12:00"), "yyyy-MM-dd HH:mm")
        ).otherwise(col("tweet_timestamp"))
    )
    .withColumn("sentiment_score",  when(col("sentiment_score").isNull(),  lit(avg_sentiment)).otherwise(col("sentiment_score")))
    .withColumn("positive_score",   when(col("positive_score").isNull(),   lit(avg_positive)).otherwise(col("positive_score")))
    .withColumn("negative_score",   when(col("negative_score").isNull(),   lit(avg_negative)).otherwise(col("negative_score")))
    .withColumn("neutral_score",    when(col("neutral_score").isNull(),    lit(avg_neutral)).otherwise(col("neutral_score")))
    .withColumn("likes",            when(col("likes").isNull(),            lit(avg_likes_sent)).otherwise(col("likes")))
    .withColumn("impressions",      when(col("impressions").isNull(),      lit(avg_impressions_sent)).otherwise(col("impressions")))
    .withColumn("engagement_count", when(col("engagement_count").isNull(), lit(avg_engagement_sent)).otherwise(col("engagement_count")))
    .filter(col("tweet_id").isNotNull())
    .dropDuplicates()
)
after_sentiment = silver_sentiment.count()
silver_sentiment.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.silver.sentiment_table")
print("sentiment_table before  : " + str(before_sentiment))
print("sentiment_table after   : " + str(after_sentiment))
print("sentiment_table removed : " + str(before_sentiment - after_sentiment))
print()


before_trends = trends_bronze.count()
silver_trends = (
    trends_bronze
    .withColumn("trend_timestamp", to_date(trim(col("trend_timestamp")), "dd-MM-yyyy"))
    .withColumn("country",
        when(col("country").isNull(),
            F.element_at(F.array([lit(c) for c in country_list]),
                (F.abs(F.hash(col("trend_timestamp").cast("string"))) % 5 + 1).cast("int"))
        ).otherwise(trim(col("country")))
    )
    .withColumn("tweet_volume",     when(col("tweet_volume").isNull(),     lit(avg_tweet_volume)).otherwise(col("tweet_volume")))
    .withColumn("mention_count",    when(col("mention_count").isNull(),    lit(avg_mention_count)).otherwise(col("mention_count")))
    .withColumn("retweet_count",    when(col("retweet_count").isNull(),    lit(avg_retweet_count)).otherwise(col("retweet_count")))
    .withColumn("trend_score",      when(col("trend_score").isNull(),      lit(avg_trend_score)).otherwise(col("trend_score")))
    .withColumn("sentiment_index",  when(col("sentiment_index").isNull(),  lit(avg_sentiment_index)).otherwise(col("sentiment_index")))
    .withColumn("impressions",      when(col("impressions").isNull(),      lit(avg_impressions_trends)).otherwise(col("impressions")))
    .withColumn("engagement_count", when(col("engagement_count").isNull(), lit(avg_engagement_trends)).otherwise(col("engagement_count")))
    .drop("topic_catagory")
    .filter(col("trend_timestamp").isNotNull())
    .dropDuplicates()
)
after_trends = silver_trends.count()
silver_trends.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.silver.trends_table")
print("trends_table before  : " + str(before_trends))
print("trends_table after   : " + str(after_trends))
print("trends_table removed : " + str(before_trends - after_trends))
print()


before_users = users_bronze.count()
silver_metadata = (
    users_bronze
    .withColumnRenamed("topic_catagory", "topic_category")
    .withColumn("country",
        when(col("country").isNull(),
            F.element_at(F.array([lit(c) for c in country_list]),
                (F.abs(F.hash(col("user_id").cast("string"))) % 5 + 1).cast("int"))
        ).otherwise(trim(col("country")))
    )
    .withColumn("topic_category",
        when(col("topic_category").isNull(),
            F.element_at(F.array([lit(t) for t in topic_list]),
                (F.abs(F.hash(col("user_id").cast("string"))) % 4 + 1).cast("int"))
        ).otherwise(trim(col("topic_category")))
    )
    .withColumn("account_created_date",
        to_timestamp(trim(col("account_created_date")), "dd-MM-yyyy HH:mm")
    )
    .withColumn("account_created_date",
        when(col("account_created_date").isNull(),
            F.to_timestamp(F.lit("2025-01-15 12:00"), "yyyy-MM-dd HH:mm")
        ).otherwise(col("account_created_date"))
    )
    .withColumn("varified",        when(col("varified").isNull(),        lit("false")).otherwise(col("varified")))
    .withColumn("followers_count", when(col("followers_count").isNull(), lit(avg_followers)).otherwise(col("followers_count")))
    .withColumn("following_count", when(col("following_count").isNull(), lit(avg_following)).otherwise(col("following_count")))
    .withColumn("likes_count",     when(col("likes_count").isNull(),     lit(avg_likes_users)).otherwise(col("likes_count")))
    .withColumn("shares_count",    when(col("shares_count").isNull(),    lit(avg_shares)).otherwise(col("shares_count")))
    .withColumn("posts_count",     when(col("posts_count").isNull(),     lit(avg_posts)).otherwise(col("posts_count")))
    .filter(col("user_id").isNotNull())
    .dropDuplicates()
)
after_users = silver_metadata.count()
silver_metadata.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.silver.user_metadata_table")
print("user_metadata_table before  : " + str(before_users))
print("user_metadata_table after   : " + str(after_users))
print("user_metadata_table removed : " + str(before_users - after_users))
print()


before_valid = valid_bronze.count()
silver_valid = (
    valid_bronze
    .withColumn("topic_category",
        when(col("topic_category").isNull(),
            F.element_at(F.array([lit(t) for t in topic_list]),
                (F.abs(F.hash(col("tweet_id"))) % 4 + 1).cast("int"))
        ).otherwise(trim(col("topic_category")))
    )
    .withColumn("tweet_text",
        when(col("tweet_text").isNull(), "No Tweet Content")
        .when(lower(col("tweet_text")) == "null", "No Tweet Content")
        .otherwise(trim(col("tweet_text")))
    )
    .withColumn("tweet_timestamp",
        to_timestamp(trim(col("tweet_timestamp")), "dd-MM-yyyy HH:mm")
    )
    .withColumn("tweet_timestamp",
        when(col("tweet_timestamp").isNull(),
            F.to_timestamp(F.lit("2025-01-15 12:00"), "yyyy-MM-dd HH:mm")
        ).otherwise(col("tweet_timestamp"))
    )
    .withColumn("impressions",      when(col("impressions").isNull(),      lit(avg_impressions_valid)).otherwise(col("impressions")))
    .withColumn("likes",            when(col("likes").isNull(),            lit(avg_likes_valid)).otherwise(col("likes")))
    .withColumn("retweets",         when(col("retweets").isNull(),         lit(avg_retweets_valid)).otherwise(col("retweets")))
    .withColumn("replies",          when(col("replies").isNull(),          lit(avg_replies_valid)).otherwise(col("replies")))
    .withColumn("engagement_count", when(col("engagement_count").isNull(), lit(avg_engagement_valid)).otherwise(col("engagement_count")))
    .withColumn("sentiment_score",  when(col("sentiment_score").isNull(),  lit(avg_sentiment_valid)).otherwise(col("sentiment_score")))
    .filter(col("tweet_id").isNotNull())
    .dropDuplicates()
)
after_valid = silver_valid.count()
silver_valid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.silver.valid_table")
print("valid_table before  : " + str(before_valid))
print("valid_table after   : " + str(after_valid))
print("valid_table removed : " + str(before_valid - after_valid))
print()

print("=" * 50)
print("FINAL SILVER ROW COUNTS")
print("=" * 50)
print("tweets_table        : " + str(spark.table("social_catalog.silver.tweets_table").count()))
print("sentiment_table     : " + str(spark.table("social_catalog.silver.sentiment_table").count()))
print("trends_table        : " + str(spark.table("social_catalog.silver.trends_table").count()))
print("user_metadata_table : " + str(spark.table("social_catalog.silver.user_metadata_table").count()))
print("valid_table         : " + str(spark.table("social_catalog.silver.valid_table").count()))
print("=" * 50)
print("Silver Layer Complete")
